# 맥락 기반 검색으로 RAG 강화하기

> 참고: 맥락 기반 검색(Contextual Retrieval)의 배경과 여러 데이터셋에 대한 추가 성능 평가는 함께 제공되는 [블로그 글](https://www.anthropic.com/news/contextual-retrieval)을 읽어 보시길 권합니다.

검색 증강 생성(RAG)을 사용하면 Claude가 응답할 때 여러분의 내부 지식 베이스, 코드베이스, 그 밖의 문서 모음을 활용할 수 있습니다. 기업들은 고객 지원, 사내 문서 질의응답, 재무·법무 분석, 코드 생성 등 여러 업무 흐름을 개선하기 위해 RAG 애플리케이션을 점점 더 많이 만들고 있습니다.

[별도 가이드](https://github.com/anthropics/anthropic-cookbook/blob/main/capabilities/retrieval_augmented_generation/guide.ipynb)에서는 기본 검색 시스템을 구성하고, 성능을 평가하는 방법을 보여 준 뒤, 성능을 개선하는 몇 가지 기법을 정리했습니다. 이 가이드에서는 검색 성능을 개선하는 기법 하나를 소개합니다. 맥락 기반 임베딩(Contextual Embeddings)입니다.

전통적인 RAG에서는 효율적인 검색을 위해 문서를 작은 청크로 나눕니다. 많은 응용에서 잘 통하는 방식이지만, 개별 청크에 맥락이 충분하지 않으면 문제가 생길 수 있습니다. 맥락 기반 임베딩은 임베딩 전에 각 청크에 관련 맥락을 더해 이 문제를 해결합니다. 임베딩되는 청크의 품질이 좋아져 더 정확한 검색과 더 나은 전체 성능으로 이어집니다. 우리가 시험한 모든 데이터 출처의 평균으로, 맥락 기반 임베딩은 상위 20개 청크 검색 실패율을 35% 줄였습니다.

같은 청크별 맥락을 BM25 검색에도 사용해 검색 성능을 더 끌어올릴 수 있습니다. 이 기법은 "맥락 기반 BM25" 절에서 소개합니다.

이 가이드에서는 코드베이스 9개를 지식 베이스로 삼아 맥락 기반 검색 시스템을 만들고 최적화하는 방법을 보여 줍니다. 다음 순서로 진행합니다.

1) 성능 기준을 세우기 위한 기본 검색 파이프라인 구성

2) 맥락 기반 임베딩: 무엇이고, 왜 통하며, 프롬프트 캐싱이 어떻게 이를 실제 프로덕션에서 쓸 만한 것으로 만드는가

3) 맥락 기반 임베딩 구현과 성능 개선 확인

4) 맥락 기반 BM25: *맥락 기반* BM25 하이브리드 검색으로 성능 개선하기

5) 재순위화로 성능 개선하기

### 평가 지표와 데이터셋

기본적인 문자 분할 방식으로 청크가 나뉜, 미리 청킹된 코드베이스 9개 데이터셋을 사용합니다. 평가 데이터셋에는 질의 248개가 들어 있고 각각 '골든 청크'를 갖고 있습니다. 성능 평가에는 Pass@k라는 지표를 사용합니다. Pass@k는 각 질의에 대해 검색된 상위 k개 문서 안에 '골든 문서'가 있었는지 확인합니다. 이 사례에서 맥락 기반 임베딩은 Pass@10 성능을 약 87%에서 약 95%로 끌어올렸습니다.

코드 파일과 그 청크는 `data/codebase_chunks.json`에, 평가 데이터셋은 `data/evaluation_set.jsonl`에 있습니다

#### 추가 참고 사항

이 검색 방식의 비용을 관리하는 데 프롬프트 캐싱이 도움이 됩니다. 현재 Anthropic 자체 API에서 사용할 수 있으며, AWS Bedrock과 GCP Vertex의 파트너 환경에도 곧 제공될 예정입니다. 많은 고객이 RAG 솔루션을 만들 때 AWS Knowledge Bases와 GCP Vertex AI API를 활용한다는 것을 알고 있으며, 이 방법은 약간의 커스터마이징으로 두 플랫폼 모두에서 사용할 수 있습니다. 이에 대한 안내가 필요하면 Anthropic이나 AWS/GCP 담당 팀에 문의해 보세요!

Bedrock에서 이 방법을 더 쉽게 쓸 수 있도록, AWS 팀이 각 문서에 맥락을 더하는 Lambda 함수를 구현할 수 있는 코드를 제공해 주었습니다. 이 Lambda 함수를 배포하면 [Bedrock Knowledge Base](https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base-create.html)를 설정할 때 커스텀 청킹 옵션으로 선택할 수 있습니다. 코드는 `contextual-rag-lambda-function`에 있고, 주 Lambda 함수 코드는 `lambda_function.py`에 있습니다.

## 목차

1) 준비

2) 기본 RAG

3) 맥락 기반 임베딩

4) 맥락 기반 BM25

5) 재순위화

## 준비

이 가이드를 시작하기 전에 다음을 확인하세요.

**기술 요건:**
- 중급 수준의 Python 프로그래밍
- RAG(검색 증강 생성)에 대한 기본 이해
- 벡터 데이터베이스와 임베딩에 대한 친숙함
- 기본적인 명령줄 사용 능력

**시스템 요구 사항:**
- Python 3.8 이상
- 설치되어 실행 중인 Docker (선택, BM25 검색용)
- 4GB 이상의 가용 RAM
- 벡터 데이터베이스용 디스크 공간 약 5~10GB

**API 접근:**
- [Anthropic API 키](https://console.anthropic.com/) (무료 등급으로 충분)
- [Voyage AI API 키](https://www.voyageai.com/)
- [Cohere API 키](https://cohere.com/)

**시간과 비용:**
- 예상 소요 시간: 30~45분
- API 비용: 전체 데이터셋을 돌리는 데 약 $5~10

### 라이브러리 

다음 라이브러리가 필요합니다.

1) `anthropic` — Claude와 상호작용

2) `voyageai` — 고품질 임베딩 생성

3) `cohere` — 재순위화

4) `elasticsearch` — 성능 좋은 BM25 검색

3) 데이터 조작과 시각화를 위한 `pandas`, `numpy`, `matplotlib`, `scikit-learn`

### 환경 변수 

다음 환경 변수가 설정되어 있는지 확인하세요.

```
- VOYAGE_API_KEY
- ANTHROPIC_API_KEY
- COHERE_API_KEY
```

In [6]:
%%capture
!pip install --upgrade anthropic voyageai cohere elasticsearch pandas numpy

새 모델이 나올 때 쉽게 바꿀 수 있도록 모델 이름을 앞부분에 정의해 둡니다

In [ ]:
MODEL_NAME = "claude-haiku-4-5"

맥락 설명 생성에 사용할 Anthropic 클라이언트를 초기화하는 것부터 시작하겠습니다.

In [5]:
import os

import anthropic

client = anthropic.Anthropic(
    # This is the default and can be omitted
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

## 벡터 DB 클래스 초기화하기

임베딩 저장과 유사도 검색을 처리할 VectorDB 클래스를 만듭니다. 이 클래스는 RAG 파이프라인에서 세 가지 핵심 역할을 합니다.

1. **임베딩 생성**: Voyage AI의 임베딩 모델로 텍스트 청크를 벡터 표현으로 변환합니다
2. **저장과 캐싱**: 임베딩을 디스크에 저장해 다시 계산하지 않게 합니다(시간과 API 비용을 아낍니다)
3. **유사도 검색**: 코사인 유사도로 주어진 질의에 가장 관련 있는 청크를 가져옵니다

이 가이드에서는 pickle 직렬화를 쓰는 간단한 메모리 벡터 데이터베이스를 사용합니다. 코드를 이해하기 쉽고 외부 의존성이 필요 없습니다. 이 클래스는 생성 후 임베딩을 디스크에 자동 저장하므로 임베딩 비용은 한 번만 내면 됩니다.

프로덕션에서는 호스팅 벡터 데이터베이스 솔루션을 고려하세요.

아래 VectorDB 클래스는 프로덕션 솔루션에서 쓰는 것과 같은 인터페이스 패턴을 따르므로 나중에 교체하기 쉽습니다. 주요 기능으로는 배치 처리(한 번에 128청크), tqdm을 통한 진행 상황 추적, 평가 중 반복 검색 속도를 높이는 질의 캐싱이 있습니다.

In [ ]:
import json
import pickle
from typing import Any

import numpy as np
import voyageai
from tqdm import tqdm


class VectorDB:
    def __init__(self, name: str, api_key=None):
        if api_key is None:
            api_key = os.getenv("VOYAGE_API_KEY")
        self.client = voyageai.Client(api_key=api_key)
        self.name = name
        self.embeddings = []
        self.metadata = []
        self.query_cache = {}
        self.db_path = f"./data/{name}/vector_db.pkl"

    def load_data(self, dataset: list[dict[str, Any]]):
        if self.embeddings and self.metadata:
            print("Vector database is already loaded. Skipping data loading.")
            return
        if os.path.exists(self.db_path):
            print("Loading vector database from disk.")
            self.load_db()
            return

        texts_to_embed = []
        metadata = []
        total_chunks = sum(len(doc["chunks"]) for doc in dataset)

        with tqdm(total=total_chunks, desc="Processing chunks") as pbar:
            for doc in dataset:
                for chunk in doc["chunks"]:
                    texts_to_embed.append(chunk["content"])
                    metadata.append(
                        {
                            "doc_id": doc["doc_id"],
                            "original_uuid": doc["original_uuid"],
                            "chunk_id": chunk["chunk_id"],
                            "original_index": chunk["original_index"],
                            "content": chunk["content"],
                        }
                    )
                    pbar.update(1)

        self._embed_and_store(texts_to_embed, metadata)
        self.save_db()

        print(f"Vector database loaded and saved. Total chunks processed: {len(texts_to_embed)}")

    def _embed_and_store(self, texts: list[str], data: list[dict[str, Any]]):
        batch_size = 128
        with tqdm(total=len(texts), desc="Embedding chunks") as pbar:
            result = []
            for i in range(0, len(texts), batch_size):
                batch = texts[i : i + batch_size]
                batch_result = self.client.embed(batch, model="voyage-2").embeddings
                result.extend(batch_result)
                pbar.update(len(batch))

        self.embeddings = result
        self.metadata = data

    def search(self, query: str, k: int = 20) -> list[dict[str, Any]]:
        if query in self.query_cache:
            query_embedding = self.query_cache[query]
        else:
            query_embedding = self.client.embed([query], model="voyage-2").embeddings[0]
            self.query_cache[query] = query_embedding

        if not self.embeddings:
            raise ValueError("No data loaded in the vector database.")

        similarities = np.dot(self.embeddings, query_embedding)
        top_indices = np.argsort(similarities)[::-1][:k]

        top_results = []
        for idx in top_indices:
            result = {
                "metadata": self.metadata[idx],
                "similarity": float(similarities[idx]),
            }
            top_results.append(result)

        return top_results

    def save_db(self):
        data = {
            "embeddings": self.embeddings,
            "metadata": self.metadata,
            "query_cache": json.dumps(self.query_cache),
        }
        os.makedirs(os.path.dirname(self.db_path), exist_ok=True)
        with open(self.db_path, "wb") as file:
            pickle.dump(data, file)

    def load_db(self):
        if not os.path.exists(self.db_path):
            raise ValueError(
                "Vector database file not found. Use load_data to create a new database."
            )
        with open(self.db_path, "rb") as file:
            data = pickle.load(file)
        self.embeddings = data["embeddings"]
        self.metadata = data["metadata"]
        self.query_cache = json.loads(data["query_cache"])

이제 이 클래스로 데이터셋을 불러올 수 있습니다

In [12]:
# Load your transformed dataset
with open("data/codebase_chunks.json") as f:
    transformed_dataset = json.load(f)

# Initialize the VectorDB
base_db = VectorDB("base_db")

# Load and process the data
base_db.load_data(transformed_dataset)

Embedding chunks: 100%|██████████| 737/737 [00:42<00:00, 17.28it/s]

Vector database loaded and saved. Total chunks processed: 737


## 기본 RAG

먼저 아주 기본적인 방식으로 RAG 파이프라인을 구성해 보겠습니다. 업계에서 흔히 '나이브 RAG'라 부르는 것입니다. 기본 RAG 파이프라인은 다음 세 단계로 이뤄집니다.

1) 문서를 제목 기준으로 청크로 나누기 — 각 소제목의 내용만 담습니다

2) 각 문서를 임베딩하기

3) 질의에 답하기 위해 코사인 유사도로 문서 검색하기

In [26]:
import json
from collections.abc import Callable
from typing import Any

from tqdm import tqdm


def load_jsonl(file_path: str) -> list[dict[str, Any]]:
    """Load JSONL file and return a list of dictionaries."""
    with open(file_path) as file:
        return [json.loads(line) for line in file]


def evaluate_retrieval(
    queries: list[dict[str, Any]], retrieval_function: Callable, db, k: int = 20
) -> dict[str, float]:
    total_score = 0
    total_queries = len(queries)

    for query_item in tqdm(queries, desc="Evaluating retrieval"):
        query = query_item["query"]
        golden_chunk_uuids = query_item["golden_chunk_uuids"]

        # Find all golden chunk contents
        golden_contents = []
        for doc_uuid, chunk_index in golden_chunk_uuids:
            golden_doc = next(
                (doc for doc in query_item["golden_documents"] if doc["uuid"] == doc_uuid), None
            )
            if not golden_doc:
                print(f"Warning: Golden document not found for UUID {doc_uuid}")
                continue

            golden_chunk = next(
                (chunk for chunk in golden_doc["chunks"] if chunk["index"] == chunk_index), None
            )
            if not golden_chunk:
                print(
                    f"Warning: Golden chunk not found for index {chunk_index} in document {doc_uuid}"
                )
                continue

            golden_contents.append(golden_chunk["content"].strip())

        if not golden_contents:
            print(f"Warning: No golden contents found for query: {query}")
            continue

        retrieved_docs = retrieval_function(query, db, k=k)

        # Count how many golden chunks are in the top k retrieved documents
        chunks_found = 0
        for golden_content in golden_contents:
            for doc in retrieved_docs[:k]:
                retrieved_content = (
                    doc["metadata"]
                    .get("original_content", doc["metadata"].get("content", ""))
                    .strip()
                )
                if retrieved_content == golden_content:
                    chunks_found += 1
                    break

        query_score = chunks_found / len(golden_contents)
        total_score += query_score

    average_score = total_score / total_queries
    pass_at_n = average_score * 100
    return {"pass_at_n": pass_at_n, "average_score": average_score, "total_queries": total_queries}


def retrieve_base(query: str, db, k: int = 20) -> list[dict[str, Any]]:
    """
    Retrieve relevant documents using either VectorDB or ContextualVectorDB.

    :param query: The query string
    :param db: The VectorDB or ContextualVectorDB instance
    :param k: Number of top results to retrieve
    :return: List of retrieved documents
    """
    return db.search(query, k=k)


def evaluate_db(db, original_jsonl_path: str, k):
    # Load the original JSONL data for queries and ground truth
    original_data = load_jsonl(original_jsonl_path)

    # Evaluate retrieval
    results = evaluate_retrieval(original_data, retrieve_base, db, k)
    return results


def evaluate_and_display(db, jsonl_path: str, k_values: list[int] = None, db_name: str = ""):
    """
    Evaluate retrieval performance across multiple k values and display formatted results.

    Args:
        db: Vector database instance (VectorDB or ContextualVectorDB)
        jsonl_path: Path to evaluation dataset
        k_values: List of k values to evaluate (default: [5, 10, 20])
        db_name: Optional name for the database being evaluated

    Returns:
        Dict mapping k values to their results
    """
    if k_values is None:
        k_values = [5, 10, 20]
    results = {}

    print(f"{'=' * 60}")
    if db_name:
        print(f"Evaluation Results: {db_name}")
    else:
        print("Evaluation Results")
    print(f"{'=' * 60}\n")

    for k in k_values:
        print(f"Evaluating Pass@{k}...")
        results[k] = evaluate_db(db, jsonl_path, k)
        print()  # Add spacing between evaluations

    # Print summary table
    print(f"{'=' * 60}")
    print(f"{'Metric':<15} {'Pass Rate':<15} {'Score':<15}")
    print(f"{'-' * 60}")
    for k in k_values:
        pass_rate = f"{results[k]['pass_at_n']:.2f}%"
        score = f"{results[k]['average_score']:.4f}"
        print(f"{'Pass@' + str(k):<15} {pass_rate:<15} {score:<15}")
    print(f"{'=' * 60}\n")

    return results

이제 기본 RAG 시스템을 평가해 기준 성능을 세우겠습니다. k=5, 10, 20에서 시험해 검색 상위 결과에 골든 청크가 얼마나 들어오는지 봅니다. 개선을 측정할 기준점이 됩니다.

In [ ]:
results = evaluate_and_display(
    base_db, "data/evaluation_set.jsonl", k_values=[5, 10, 20], db_name="Baseline RAG"
)

Evaluation Results: Contextual Embeddings

Evaluating Pass@5...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 65.26it/s]



Evaluating Pass@10...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.87it/s]



Evaluating Pass@20...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.72it/s]


Metric          Pass Rate       Score          
------------------------------------------------------------
Pass@5          80.92%          0.8092         
Pass@10         87.15%          0.8715         
Pass@20         90.06%          0.9006         



이 결과가 기준 RAG 성능입니다. 시스템이 상위 5개 결과에서 81%, 상위 10개에서 87%, 상위 20개에서 90%의 비율로 올바른 청크를 찾아냅니다.

## 맥락 기반 임베딩

기본 RAG에서는 개별 청크가 홀로 임베딩될 때 맥락이 부족한 경우가 많습니다. 맥락 기반 임베딩은 Claude로 각 청크를 원본 문서 안에 "자리매김"해 주는 짧은 설명을 생성해 이를 해결합니다. 그런 다음 청크와 이 맥락을 함께 임베딩해 더 풍부한 벡터 표현을 만듭니다.

코드베이스 데이터셋의 청크마다 그 청크와 원본 파일 전체를 Claude에 전달합니다. Claude는 그 청크에 무엇이 담겨 있고 파일 전체에서 어디에 위치하는지 간결하게 설명합니다. 이 맥락이 임베딩 전에 청크 앞에 붙습니다.

### 비용과 지연 시간 고려 사항

**이 비용은 언제 발생하나요?** 맥락화는 질의할 때마다가 아니라 데이터를 적재할 때 한 번 일어납니다. 검색할 때마다 지연을 더하는 HyDE(가상 문서 임베딩) 같은 기법과 달리, 맥락 기반 임베딩은 벡터 데이터베이스를 만들 때의 일회성 비용입니다. 프롬프트 캐싱이 이를 실용적으로 만들어 줍니다. 같은 문서의 모든 청크를 순차적으로 처리하므로 프롬프트 캐싱으로 상당한 비용을 아낄 수 있습니다.

1. 첫 청크: 문서 전체를 캐시에 씁니다(약간의 프리미엄을 지불)
2. 이후 청크: 문서를 캐시에서 읽습니다(해당 토큰에 90% 할인)
3. 캐시는 5분간 유지되며, 한 문서의 모든 청크를 처리하기에 충분합니다

**비용 예시**: 8k 토큰 문서 안의 800토큰 청크에 100토큰의 맥락을 생성하는 경우, 총비용은 문서 100만 토큰당 $1.02입니다. 아래 코드를 실행하면 로그에서 캐시 절감 효과를 확인할 수 있습니다.

**참고:** 일부 임베딩 모델에는 고정된 입력 토큰 한도가 있습니다. 맥락 기반 임베딩으로 성능이 오히려 나빠졌다면 맥락화된 청크가 잘리고 있을 수 있으니, 컨텍스트 윈도가 더 큰 임베딩 모델을 고려해 보세요.

--- 

청크 하나에 대한 맥락을 생성해 맥락 기반 임베딩이 어떻게 동작하는지 살펴보겠습니다. Claude로 자리매김 맥락을 만들고, 프롬프트 캐싱 지표도 함께 확인합니다.

In [ ]:
DOCUMENT_CONTEXT_PROMPT = """
<document>
{doc_content}
</document>
"""

CHUNK_CONTEXT_PROMPT = """
Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>

Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else.
"""


def situate_context(doc: str, chunk: str) -> str:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": DOCUMENT_CONTEXT_PROMPT.format(doc_content=doc),
                        "cache_control": {
                            "type": "ephemeral"
                        },  # we will make use of prompt caching for the full documents
                    },
                    {
                        "type": "text",
                        "text": CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk),
                    },
                ],
            }
        ],
    )
    return response


jsonl_data = load_jsonl("data/evaluation_set.jsonl")
# Example usage
doc_content = jsonl_data[0]["golden_documents"][0]["content"]
chunk_content = jsonl_data[0]["golden_chunks"][0]["content"]

response = situate_context(doc_content, chunk_content)
print(f"Situated context: {response.content[0].text}")
print("-" * 10)
# Print cache performance metrics
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
print(f"Cache creation input tokens: {response.usage.cache_creation_input_tokens}")
print(f"Cache read input tokens: {response.usage.cache_read_input_tokens}")

Situated context: This chunk contains the module documentation and initial struct definition for a differential fuzzing executor. It introduces the `DiffExecutor` struct that wraps two executors (primary and secondary) to run them sequentially with the same input, comparing their behavior for differential testing. The chunk establishes the core data structure and imports needed for the differential fuzzing implementation.
----------
Input tokens: 3412
Output tokens: 76
Cache creation input tokens: 0
Cache read input tokens: 0


### 맥락 기반 벡터 데이터베이스 만들기

개별 청크에 대한 맥락 설명을 생성하는 방법을 봤으니, 이제 데이터셋 전체를 처리하도록 규모를 키우겠습니다. 아래 `ContextualVectorDB` 클래스는 적재 과정에서 자동 맥락화를 수행하도록 기본 `VectorDB`를 확장합니다.

**주요 기능:**

- **병렬 처리**: ThreadPoolExecutor로 여러 청크를 동시에 맥락화합니다(스레드 수 설정 가능)
- **자동 프롬프트 캐싱**: 캐시 적중을 극대화하도록 문서 단위로 청크를 처리합니다
- **토큰 추적**: 캐시 성능을 모니터링하고 실제 절감액을 계산합니다
- **영속 저장**: 임베딩과 맥락화된 메타데이터를 모두 디스크에 저장합니다

실행하면 토큰 사용 통계를 눈여겨보세요. 입력 토큰의 70~80%가 캐시에서 읽히는 것을 볼 수 있고, 프롬프트 캐싱의 극적인 비용 절감을 보여 줍니다. 737청크 데이터셋에서 약 $15가 들었을 적재 작업이 약 $3로 줄어듭니다.

In [ ]:
import json
import os
import pickle
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any

import anthropic
import numpy as np
import voyageai
from tqdm import tqdm


class ContextualVectorDB:
    def __init__(self, name: str, voyage_api_key=None, anthropic_api_key=None):
        if voyage_api_key is None:
            voyage_api_key = os.getenv("VOYAGE_API_KEY")
        if anthropic_api_key is None:
            anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

        self.voyage_client = voyageai.Client(api_key=voyage_api_key)
        self.anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)
        self.name = name
        self.embeddings = []
        self.metadata = []
        self.query_cache = {}
        self.db_path = f"./data/{name}/contextual_vector_db.pkl"

        self.token_counts = {"input": 0, "output": 0, "cache_read": 0, "cache_creation": 0}
        self.token_lock = threading.Lock()

    def situate_context(self, doc: str, chunk: str) -> tuple[str, Any]:
        DOCUMENT_CONTEXT_PROMPT = """
        <document>
        {doc_content}
        </document>
        """

        CHUNK_CONTEXT_PROMPT = """
        Here is the chunk we want to situate within the whole document
        <chunk>
        {chunk_content}
        </chunk>

        Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
        Answer only with the succinct context and nothing else.
        """

        response = self.anthropic_client.messages.create(
            model=MODEL_NAME,
            max_tokens=1000,
            temperature=0.0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": DOCUMENT_CONTEXT_PROMPT.format(doc_content=doc),
                            "cache_control": {
                                "type": "ephemeral"
                            },  # we will make use of prompt caching for the full documents
                        },
                        {
                            "type": "text",
                            "text": CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk),
                        },
                    ],
                },
            ],
            extra_headers={"anthropic-beta": "prompt-caching-2024-07-31"},
        )
        return response.content[0].text, response.usage

    def load_data(self, dataset: list[dict[str, Any]], parallel_threads: int = 1):
        if self.embeddings and self.metadata:
            print("Vector database is already loaded. Skipping data loading.")
            return
        if os.path.exists(self.db_path):
            print("Loading vector database from disk.")
            self.load_db()
            return

        texts_to_embed = []
        metadata = []
        total_chunks = sum(len(doc["chunks"]) for doc in dataset)

        def process_chunk(doc, chunk):
            # for each chunk, produce the context
            contextualized_text, usage = self.situate_context(doc["content"], chunk["content"])
            with self.token_lock:
                self.token_counts["input"] += usage.input_tokens
                self.token_counts["output"] += usage.output_tokens
                self.token_counts["cache_read"] += usage.cache_read_input_tokens
                self.token_counts["cache_creation"] += usage.cache_creation_input_tokens

            return {
                # append the context to the original text chunk
                "text_to_embed": f"{contextualized_text}\n\n{chunk['content']}",
                "metadata": {
                    "doc_id": doc["doc_id"],
                    "original_uuid": doc["original_uuid"],
                    "chunk_id": chunk["chunk_id"],
                    "original_index": chunk["original_index"],
                    "original_content": chunk["content"],
                    "contextualized_content": contextualized_text,
                },
            }

        print(f"Processing {total_chunks} chunks with {parallel_threads} threads")
        with ThreadPoolExecutor(max_workers=parallel_threads) as executor:
            futures = []
            for doc in dataset:
                for chunk in doc["chunks"]:
                    futures.append(executor.submit(process_chunk, doc, chunk))

            for future in tqdm(as_completed(futures), total=total_chunks, desc="Processing chunks"):
                result = future.result()
                texts_to_embed.append(result["text_to_embed"])
                metadata.append(result["metadata"])

        self._embed_and_store(texts_to_embed, metadata)
        self.save_db()

        # logging token usage
        print(
            f"Contextual Vector database loaded and saved. Total chunks processed: {len(texts_to_embed)}"
        )
        print(f"Total input tokens without caching: {self.token_counts['input']}")
        print(f"Total output tokens: {self.token_counts['output']}")
        print(f"Total input tokens written to cache: {self.token_counts['cache_creation']}")
        print(f"Total input tokens read from cache: {self.token_counts['cache_read']}")

        total_tokens = (
            self.token_counts["input"]
            + self.token_counts["cache_read"]
            + self.token_counts["cache_creation"]
        )
        savings_percentage = (
            (self.token_counts["cache_read"] / total_tokens) * 100 if total_tokens > 0 else 0
        )
        print(
            f"Total input token savings from prompt caching: {savings_percentage:.2f}% of all input tokens used were read from cache."
        )
        print("Tokens read from cache come at a 90 percent discount!")

    # we use voyage AI here for embeddings. Read more here: https://docs.voyageai.com/docs/embeddings
    def _embed_and_store(self, texts: list[str], data: list[dict[str, Any]]):
        batch_size = 128
        result = [
            self.voyage_client.embed(texts[i : i + batch_size], model="voyage-2").embeddings
            for i in range(0, len(texts), batch_size)
        ]
        self.embeddings = [embedding for batch in result for embedding in batch]
        self.metadata = data

    def search(self, query: str, k: int = 20) -> list[dict[str, Any]]:
        if query in self.query_cache:
            query_embedding = self.query_cache[query]
        else:
            query_embedding = self.voyage_client.embed([query], model="voyage-2").embeddings[0]
            self.query_cache[query] = query_embedding

        if not self.embeddings:
            raise ValueError("No data loaded in the vector database.")

        similarities = np.dot(self.embeddings, query_embedding)
        top_indices = np.argsort(similarities)[::-1][:k]

        top_results = []
        for idx in top_indices:
            result = {
                "metadata": self.metadata[idx],
                "similarity": float(similarities[idx]),
            }
            top_results.append(result)
        return top_results

    def save_db(self):
        data = {
            "embeddings": self.embeddings,
            "metadata": self.metadata,
            "query_cache": json.dumps(self.query_cache),
        }
        os.makedirs(os.path.dirname(self.db_path), exist_ok=True)
        with open(self.db_path, "wb") as file:
            pickle.dump(data, file)

    def load_db(self):
        if not os.path.exists(self.db_path):
            raise ValueError(
                "Vector database file not found. Use load_data to create a new database."
            )
        with open(self.db_path, "rb") as file:
            data = pickle.load(file)
        self.embeddings = data["embeddings"]
        self.metadata = data["metadata"]
        self.query_cache = json.loads(data["query_cache"])

In [22]:
# Load the transformed dataset
with open("data/codebase_chunks.json") as f:
    transformed_dataset = json.load(f)

# Initialize the ContextualVectorDB
contextual_db = ContextualVectorDB("my_contextual_db")

# Load and process the data
# note: consider increasing the number of parallel threads to run this faster, or reducing the number of parallel threads if concerned about hitting your API rate limit
contextual_db.load_data(transformed_dataset, parallel_threads=5)

Processing 737 chunks with 5 threads


Processing chunks: 100%|██████████| 737/737 [05:32<00:00,  2.22it/s]


Contextual Vector database loaded and saved. Total chunks processed: 737
Total input tokens without caching: 1223730
Total output tokens: 58161
Total input tokens written to cache: 176079
Total input tokens read from cache: 2267069
Total input token savings from prompt caching: 61.83% of all input tokens used were read from cache.
Tokens read from cache come at a 90 percent discount!


이 수치들이 맥락 기반 임베딩에서 프롬프트 캐싱의 힘을 보여 줍니다.

- 코드베이스 파일 9개에 걸쳐 **청크 737개**를 처리했습니다
- **입력 토큰의 61.83%**가 캐시에서 읽혔습니다(227만 토큰에 90% 할인)
- 캐싱이 없었다면 입력 토큰 비용이 **약 $9.20**이었을 것입니다
- 캐싱을 쓰면 실제 비용이 **약 $2.85**로 떨어집니다(69% 절감)

캐시 적중률은 문서마다 청크가 몇 개인지에 달려 있습니다. 청크가 많은 파일일수록 캐싱 효과가 큽니다. 문서 전체를 캐시에 한 번 쓰고 그 파일의 청크마다 반복해서 읽기 때문입니다. 청크를 무작위로 섞지 않고 문서 단위로 순차 처리하는 것이 캐시 효율을 극대화하는 데 결정적인 이유입니다.

이제 이 맥락화가 기준점 대비 검색 성능을 얼마나 개선하는지 평가해 보겠습니다.

In [28]:
results = evaluate_and_display(
    contextual_db,
    "data/evaluation_set.jsonl",
    k_values=[5, 10, 20],
    db_name="Contextual Embeddings",
)

Evaluation Results: Contextual Embeddings

Evaluating Pass@5...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.58it/s]



Evaluating Pass@10...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.37it/s]



Evaluating Pass@20...


Evaluating retrieval: 100%|██████████| 248/248 [00:03<00:00, 64.14it/s]


Metric          Pass Rate       Score          
------------------------------------------------------------
Pass@5          88.12%          0.8812         
Pass@10         92.34%          0.9234         
Pass@20         94.29%          0.9429         



임베딩 전에 각 청크에 맥락을 더함으로써 모든 k 값에서 검색 실패를 **약 30~40%** 줄였습니다. 상위 검색 청크에 관련 없는 결과가 줄어든다는 뜻이고, 이 청크들을 Claude에 넘겨 최종 응답을 생성할 때 더 나은 답으로 이어집니다.

개선은 정밀도가 가장 중요한 Pass@5에서 가장 두드러집니다. 맥락화된 청크가 더 자주 검색될 뿐 아니라, 관련이 있을 때 더 높은 순위에 오른다는 뜻입니다.

## 맥락 기반 BM25: 하이브리드 검색

맥락 기반 임베딩만으로 Pass@10이 87%에서 92%로 올랐습니다. 시맨틱 검색과 키워드 기반 검색을 결합한 **맥락 기반 BM25**로 성능을 더 끌어올릴 수 있습니다. 검색 실패율을 한층 더 낮추는 하이브리드 접근입니다.

### 왜 하이브리드 검색인가?

시맨틱 검색은 의미와 맥락을 이해하는 데 뛰어나지만 정확한 키워드 일치를 놓칠 수 있습니다. BM25(확률적 키워드 순위 알고리즘)는 특정 용어를 찾는 데 뛰어나지만 의미를 이해하지 못합니다. 둘을 결합하면 양쪽의 장점을 모두 얻습니다.

- **시맨틱 검색**: 개념적 유사성과 바꿔 쓴 표현을 포착합니다
- **BM25**: 정확한 용어, 함수 이름, 특정 표현을 잡아냅니다
- **역순위 융합(Reciprocal Rank Fusion)**: 양쪽 결과를 지능적으로 병합합니다

### BM25란?

BM25는 문서 길이와 용어 포화를 고려해 TF-IDF를 개선한 확률적 순위 함수입니다. 키워드 관련성 순위를 잘 매기기 때문에 Elasticsearch를 비롯한 실제 검색 엔진에서 널리 쓰입니다. 기술적 세부는 [이 블로그 글](https://www.elastic.co/blog/practical-bm25-part-2-the-bm25-algorithm-and-its-variables)을 참고하세요.

원시 청크 내용만 검색하는 대신, 청크와 앞서 생성한 맥락 설명을 *모두* 검색합니다. 즉 BM25가 원본 텍스트나 설명 맥락 어느 쪽의 키워드든 매칭할 수 있습니다.

### 준비: Elasticsearch 실행하기

아래 코드를 실행하기 전에 로컬에서 Elasticsearch가 돌고 있어야 합니다. Docker가 가장 쉽습니다:

```bash
docker run -d --name elasticsearch -p 9200:9200 -p 9300:9300 \
  -e "discovery.type=single-node" \
  -e "xpack.security.enabled=false" \
  elasticsearch:9.2.0
```

## 문제 해결
- 실행 확인: docker ps | grep elasticsearch
- 9200 포트가 사용 중이라면: docker stop elasticsearch && docker rm elasticsearch
- 문제가 생기면 로그 확인: docker logs elasticsearch

## 하이브리드 검색의 동작 방식

아래 retrieve_advanced 함수는 세 단계로 동작합니다.

1. 후보 검색: 시맨틱 검색과 BM25 양쪽에서 상위 150개 결과를 가져옵니다
2. 점수 융합: 가중 역순위 융합으로 순위를 결합합니다
    - 기본값: 시맨틱 검색 80%, BM25 20% 가중치
    - 이 가중치는 조정 가능하므로 사용 사례에 맞게 실험해 보세요
3. 상위 k개 반환: 융합 후 가장 높은 점수의 결과를 선택합니다

가중치 체계 덕분에 데이터 특성에 따라 의미 이해와 키워드 정밀도 사이의 균형을 조정할 수 있습니다.

In [ ]:
import json
import os
from typing import Any

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from tqdm import tqdm


class ElasticsearchBM25:
    def __init__(self, index_name: str = "contextual_bm25_index"):
        self.es_client = Elasticsearch("http://localhost:9200")
        self.index_name = index_name
        self.create_index()

    def create_index(self):
        index_settings = {
            "settings": {
                "analysis": {"analyzer": {"default": {"type": "english"}}},
                "similarity": {"default": {"type": "BM25"}},
                "index.queries.cache.enabled": False,
            },
            "mappings": {
                "properties": {
                    "content": {"type": "text", "analyzer": "english"},
                    "contextualized_content": {"type": "text", "analyzer": "english"},
                    "doc_id": {"type": "keyword", "index": False},
                    "chunk_id": {"type": "keyword", "index": False},
                    "original_index": {"type": "integer", "index": False},
                }
            },
        }

        # Change this line - remove 'body=' parameter
        if not self.es_client.indices.exists(index=self.index_name):
            self.es_client.indices.create(
                index=self.index_name,
                settings=index_settings["settings"],
                mappings=index_settings["mappings"],
            )
            print(f"Created index: {self.index_name}")

    def index_documents(self, documents: list[dict[str, Any]]):
        actions = [
            {
                "_index": self.index_name,
                "_source": {
                    "content": doc["original_content"],
                    "contextualized_content": doc["contextualized_content"],
                    "doc_id": doc["doc_id"],
                    "chunk_id": doc["chunk_id"],
                    "original_index": doc["original_index"],
                },
            }
            for doc in documents
        ]
        success, _ = bulk(self.es_client, actions)
        self.es_client.indices.refresh(index=self.index_name)
        return success

    def search(self, query: str, k: int = 20) -> list[dict[str, Any]]:
        self.es_client.indices.refresh(index=self.index_name)

        # Change this - remove 'body=' and pass query directly
        response = self.es_client.search(
            index=self.index_name,
            query={
                "multi_match": {
                    "query": query,
                    "fields": ["content", "contextualized_content"],
                }
            },
            size=k,
        )

        return [
            {
                "doc_id": hit["_source"]["doc_id"],
                "original_index": hit["_source"]["original_index"],
                "content": hit["_source"]["content"],
                "contextualized_content": hit["_source"]["contextualized_content"],
                "score": hit["_score"],
            }
            for hit in response["hits"]["hits"]
        ]


def create_elasticsearch_bm25_index(db: ContextualVectorDB):
    es_bm25 = ElasticsearchBM25()
    es_bm25.index_documents(db.metadata)
    return es_bm25


def retrieve_advanced(
    query: str,
    db: ContextualVectorDB,
    es_bm25: ElasticsearchBM25,
    k: int,
    semantic_weight: float = 0.8,
    bm25_weight: float = 0.2,
):
    num_chunks_to_recall = 150

    # Semantic search
    semantic_results = db.search(query, k=num_chunks_to_recall)
    ranked_chunk_ids = [
        (result["metadata"]["doc_id"], result["metadata"]["original_index"])
        for result in semantic_results
    ]

    # BM25 search using Elasticsearch
    bm25_results = es_bm25.search(query, k=num_chunks_to_recall)
    ranked_bm25_chunk_ids = [
        (result["doc_id"], result["original_index"]) for result in bm25_results
    ]

    # Combine results
    chunk_ids = list(set(ranked_chunk_ids + ranked_bm25_chunk_ids))
    chunk_id_to_score = {}

    # Initial scoring with weights
    for chunk_id in chunk_ids:
        score = 0
        if chunk_id in ranked_chunk_ids:
            index = ranked_chunk_ids.index(chunk_id)
            score += semantic_weight * (1 / (index + 1))  # Weighted 1/n scoring for semantic
        if chunk_id in ranked_bm25_chunk_ids:
            index = ranked_bm25_chunk_ids.index(chunk_id)
            score += bm25_weight * (1 / (index + 1))  # Weighted 1/n scoring for BM25
        chunk_id_to_score[chunk_id] = score

    # Sort chunk IDs by their scores in descending order
    sorted_chunk_ids = sorted(
        chunk_id_to_score.keys(), key=lambda x: (chunk_id_to_score[x], x[0], x[1]), reverse=True
    )

    # Assign new scores based on the sorted order
    for index, chunk_id in enumerate(sorted_chunk_ids):
        chunk_id_to_score[chunk_id] = 1 / (index + 1)

    # Prepare the final results
    final_results = []
    semantic_count = 0
    bm25_count = 0
    for chunk_id in sorted_chunk_ids[:k]:
        chunk_metadata = next(
            chunk
            for chunk in db.metadata
            if chunk["doc_id"] == chunk_id[0] and chunk["original_index"] == chunk_id[1]
        )
        is_from_semantic = chunk_id in ranked_chunk_ids
        is_from_bm25 = chunk_id in ranked_bm25_chunk_ids
        final_results.append(
            {
                "chunk": chunk_metadata,
                "score": chunk_id_to_score[chunk_id],
                "from_semantic": is_from_semantic,
                "from_bm25": is_from_bm25,
            }
        )

        if is_from_semantic and not is_from_bm25:
            semantic_count += 1
        elif is_from_bm25 and not is_from_semantic:
            bm25_count += 1
        else:  # it's in both
            semantic_count += 0.5
            bm25_count += 0.5

    return final_results, semantic_count, bm25_count


def evaluate_db_advanced(
    db: ContextualVectorDB,
    original_jsonl_path: str,
    k_values: list[int] = None,
    db_name: str = "Hybrid Search",
):
    """
    Evaluate hybrid search (semantic + BM25) at multiple k values with formatted results.

    Args:
        db: ContextualVectorDB instance
        original_jsonl_path: Path to evaluation dataset
        k_values: List of k values to evaluate (default: [5, 10, 20])
        db_name: Name for the evaluation display

    Returns:
        Dict mapping k values to their results and source breakdowns
    """
    if k_values is None:
        k_values = [5, 10, 20]
    original_data = load_jsonl(original_jsonl_path)
    es_bm25 = create_elasticsearch_bm25_index(db)
    results = {}

    print(f"{'=' * 70}")
    print(f"Evaluation Results: {db_name}")
    print(f"{'=' * 70}\n")

    try:
        # Warm-up queries
        warm_up_queries = original_data[:10]
        for query_item in warm_up_queries:
            _ = retrieve_advanced(query_item["query"], db, es_bm25, k_values[0])

        for k in k_values:
            print(f"Evaluating Pass@{k}...")

            total_score = 0
            total_semantic_count = 0
            total_bm25_count = 0
            total_results = 0

            for query_item in tqdm(original_data, desc=f"Pass@{k}"):
                query = query_item["query"]
                golden_chunk_uuids = query_item["golden_chunk_uuids"]

                golden_contents = []
                for doc_uuid, chunk_index in golden_chunk_uuids:
                    golden_doc = next(
                        (doc for doc in query_item["golden_documents"] if doc["uuid"] == doc_uuid),
                        None,
                    )
                    if golden_doc:
                        golden_chunk = next(
                            (
                                chunk
                                for chunk in golden_doc["chunks"]
                                if chunk["index"] == chunk_index
                            ),
                            None,
                        )
                        if golden_chunk:
                            golden_contents.append(golden_chunk["content"].strip())

                if not golden_contents:
                    continue

                retrieved_docs, semantic_count, bm25_count = retrieve_advanced(
                    query, db, es_bm25, k
                )

                chunks_found = 0
                for golden_content in golden_contents:
                    for doc in retrieved_docs[:k]:
                        retrieved_content = doc["chunk"]["original_content"].strip()
                        if retrieved_content == golden_content:
                            chunks_found += 1
                            break

                query_score = chunks_found / len(golden_contents)
                total_score += query_score

                total_semantic_count += semantic_count
                total_bm25_count += bm25_count
                total_results += len(retrieved_docs)

            total_queries = len(original_data)
            average_score = total_score / total_queries
            pass_at_n = average_score * 100

            semantic_percentage = (
                (total_semantic_count / total_results) * 100 if total_results > 0 else 0
            )
            bm25_percentage = (total_bm25_count / total_results) * 100 if total_results > 0 else 0

            results[k] = {
                "pass_at_n": pass_at_n,
                "average_score": average_score,
                "total_queries": total_queries,
                "semantic_percentage": semantic_percentage,
                "bm25_percentage": bm25_percentage,
            }

            print(f"Pass@{k}: {pass_at_n:.2f}%")
            print(f"Semantic: {semantic_percentage:.1f}% | BM25: {bm25_percentage:.1f}%\n")

        # Print summary table
        print(f"{'=' * 70}")
        print(f"{'Metric':<12} {'Pass Rate':<12} {'Score':<12} {'Semantic':<12} {'BM25':<12}")
        print(f"{'-' * 70}")
        for k in k_values:
            r = results[k]
            print(
                f"{'Pass@' + str(k):<12} {r['pass_at_n']:>10.2f}% {r['average_score']:>10.4f} "
                f"{r['semantic_percentage']:>10.1f}% {r['bm25_percentage']:>10.1f}%"
            )
        print(f"{'=' * 70}\n")

        return results

    finally:
        # Delete the Elasticsearch index
        if es_bm25.es_client.indices.exists(index=es_bm25.index_name):
            es_bm25.es_client.indices.delete(index=es_bm25.index_name)
            print(f"Deleted Elasticsearch index: {es_bm25.index_name}")

In [39]:
results = evaluate_db_advanced(
    contextual_db,
    "data/evaluation_set.jsonl",
    k_values=[5, 10, 20],
    db_name="Contextual BM25 Hybrid Search",
)

Created index: contextual_bm25_index
Evaluation Results: Contextual BM25 Hybrid Search

Evaluating Pass@5...


Pass@5: 100%|██████████| 248/248 [00:05<00:00, 41.79it/s]


Pass@5: 88.86%
Semantic: 54.6% | BM25: 45.4%

Evaluating Pass@10...


Pass@10: 100%|██████████| 248/248 [00:05<00:00, 42.20it/s]


Pass@10: 92.31%
Semantic: 57.6% | BM25: 42.4%

Evaluating Pass@20...


Pass@20: 100%|██████████| 248/248 [00:05<00:00, 42.15it/s]


Pass@20: 95.23%
Semantic: 60.8% | BM25: 39.2%

Metric       Pass Rate    Score        Semantic     BM25        
----------------------------------------------------------------------
Pass@5            88.86%     0.8886       54.6%       45.4%
Pass@10           92.31%     0.9231       57.6%       42.4%
Pass@20           95.23%     0.9523       60.8%       39.2%

Deleted Elasticsearch index: contextual_bm25_index


## 재순위화

하이브리드 검색으로 좋은 결과(Pass@10 93.21%)를 얻었지만, 성능을 조금 더 짜낼 수 있는 기법이 하나 더 있습니다. **재순위화**입니다.

### 재순위화란?

재순위화는 두 단계로 이뤄진 검색 방식입니다.

1. **1단계 — 폭넓은 검색**: 필요한 것보다 많은 후보를 가져와 그물을 넓게 던집니다(예: 청크 100개 검색)
2. **2단계 — 정밀 선별**: 전용 재순위화 모델로 이 후보들에 점수를 매겨 가장 관련 있는 상위 k개만 고릅니다

**왜 이것이 통할까요?** 초기 검색 방법(임베딩, BM25)은 수백만 문서를 빠르게 훑도록 최적화되어 있습니다. 재순위화 모델은 더 느리지만 더 정확합니다. 작은 후보 집합에 대해서는 더 깊이 분석할 여유가 있기 때문입니다. 실무에서 잘 통하는 속도/정확도 트레이드오프입니다.

### 우리의 재순위화 방식

이 예제에서는 전체 하이브리드 검색이 아니라 맥락 기반 임베딩 위에만 얹은 더 단순한 재순위화 파이프라인을 사용합니다. 과정은 다음과 같습니다.

1. **초과 검색**: 필요한 것보다 10배 많은 결과를 가져옵니다(예: 10개가 필요하면 100개 검색)
2. **Cohere로 재순위화**: Cohere의 `rerank-english-v3.0` 모델로 모든 후보에 점수를 매깁니다
3. **상위 k개 선택**: 점수가 가장 높은 결과만 반환합니다

재순위화 모델은 원본 청크 내용과 우리가 생성한 맥락 설명을 모두 볼 수 있어, 풍부한 정보를 바탕으로 정밀하게 관련성을 판단합니다.

### 기대 성능

재순위화를 더하면 크지는 않지만 의미 있는 개선이 나타납니다.
- **재순위화 없음**: Pass@10 92.34%(맥락 기반 임베딩만)
- **재순위화 적용**: Pass@10 약 95%(2~3%p 추가 향상)

작아 보일 수 있지만, 실제 시스템에서 실패율을 7.66%에서 약 5%로 줄이면 사용자 경험이 크게 좋아집니다. 대가는 질의 지연입니다. 후보 집합 크기에 따라 질의당 100~200ms가 더해집니다.

In [ ]:
import json
from collections.abc import Callable
from typing import Any

import cohere
from tqdm import tqdm


def evaluate_db_rerank(
    db, original_jsonl_path: str, k_values: list[int] = None, db_name: str = "Reranking"
):
    """
    Evaluate reranking performance at multiple k values with formatted results.

    Args:
        db: ContextualVectorDB instance
        original_jsonl_path: Path to evaluation dataset
        k_values: List of k values to evaluate (default: [5, 10, 20])
        db_name: Name for the evaluation display

    Returns:
        Dict mapping k values to their results
    """
    if k_values is None:
        k_values = [5, 10, 20]
    original_data = load_jsonl(original_jsonl_path)
    co = cohere.Client(os.getenv("COHERE_API_KEY"))
    results = {}

    print(f"{'=' * 60}")
    print(f"Evaluation Results: {db_name}")
    print(f"{'=' * 60}\n")

    for k in k_values:
        print(f"Evaluating Pass@{k} with reranking...")

        total_score = 0
        total_queries = len(original_data)

        for query_item in tqdm(original_data, desc=f"Pass@{k}"):
            query = query_item["query"]
            golden_chunk_uuids = query_item["golden_chunk_uuids"]

            # Find golden contents
            golden_contents = []
            for doc_uuid, chunk_index in golden_chunk_uuids:
                golden_doc = next(
                    (doc for doc in query_item["golden_documents"] if doc["uuid"] == doc_uuid), None
                )
                if golden_doc:
                    golden_chunk = next(
                        (chunk for chunk in golden_doc["chunks"] if chunk["index"] == chunk_index),
                        None,
                    )
                    if golden_chunk:
                        golden_contents.append(golden_chunk["content"].strip())

            if not golden_contents:
                continue

            # Retrieve and rerank
            semantic_results = db.search(query, k=k * 10)

            # Prepare documents for reranking
            documents = [
                f"{res['metadata']['original_content']}\n\nContext: {res['metadata']['contextualized_content']}"
                for res in semantic_results
            ]

            # Rerank
            rerank_response = co.rerank(
                model="rerank-english-v3.0", query=query, documents=documents, top_n=k
            )
            time.sleep(0.1)  # Rate limiting

            # Get final results
            retrieved_docs = []
            for r in rerank_response.results:
                original_result = semantic_results[r.index]
                retrieved_docs.append(
                    {"chunk": original_result["metadata"], "score": r.relevance_score}
                )

            # Check if golden chunks are in results
            chunks_found = 0
            for golden_content in golden_contents:
                for doc in retrieved_docs[:k]:
                    retrieved_content = doc["chunk"]["original_content"].strip()
                    if retrieved_content == golden_content:
                        chunks_found += 1
                        break

            query_score = chunks_found / len(golden_contents)
            total_score += query_score

        average_score = total_score / total_queries
        pass_at_n = average_score * 100

        results[k] = {
            "pass_at_n": pass_at_n,
            "average_score": average_score,
            "total_queries": total_queries,
        }

        print(f"Pass@{k}: {pass_at_n:.2f}%")
        print(f"Average Score: {average_score:.4f}\n")

    # Print summary table
    print(f"{'=' * 60}")
    print(f"{'Metric':<15} {'Pass Rate':<15} {'Score':<15}")
    print(f"{'-' * 60}")
    for k in k_values:
        pass_rate = f"{results[k]['pass_at_n']:.2f}%"
        score = f"{results[k]['average_score']:.4f}"
        print(f"{'Pass@' + str(k):<15} {pass_rate:<15} {score:<15}")
    print(f"{'=' * 60}\n")

    return results

In [48]:
results = evaluate_db_rerank(
    contextual_db,
    "data/evaluation_set.jsonl",
    k_values=[5, 10, 20],
    db_name="Contextual Embeddings + Reranking",
)

Evaluation Results: Contextual Embeddings + Reranking

Evaluating Pass@5 with reranking...


Pass@5: 100%|██████████| 248/248 [01:40<00:00,  2.47it/s]


Pass@5: 92.15%
Average Score: 0.9215

Evaluating Pass@10 with reranking...


Pass@10: 100%|██████████| 248/248 [02:29<00:00,  1.66it/s]


Pass@10: 95.26%
Average Score: 0.9526

Evaluating Pass@20 with reranking...


Pass@20: 100%|██████████| 248/248 [03:03<00:00,  1.35it/s]

Pass@20: 97.45%
Average Score: 0.9745

Metric          Pass Rate       Score          
------------------------------------------------------------
Pass@5          92.15%          0.9215         
Pass@10         95.26%          0.9526         
Pass@20         97.45%          0.9745         




재순위화가 가장 좋은 결과를 내며 검색 실패를 거의 없앴습니다. 각 기법이 앞선 것 위에 어떻게 쌓여 이 개선에 이르렀는지 살펴보겠습니다.

기준 RAG 시스템의 Pass@10 87%에서 시작해, 고급 검색 기법을 체계적으로 적용해 95%를 넘어섰습니다. 각 방법이 서로 다른 약점을 해결합니다. 맥락 기반 임베딩은 "고립된 청크" 문제를, 하이브리드 검색은 임베딩이 놓치는 키워드 중심 질의를, 재순위화는 더 정교한 관련성 점수화로 최종 선별을 다듬습니다.

| 방식 | Pass@5 | Pass@10 | Pass@20 |
|----------|--------|---------|---------|
| **기준 RAG** | 80.92% | 87.15% | 90.06% |
| **+ 맥락 기반 임베딩** | 88.12% | 92.34% | 94.29% |
| **+ 하이브리드 검색(BM25)** | 86.43% | 93.21% | 94.99% |
| **+ 재순위화** | 92.15% | 95.26% | 97.45% |

**핵심 정리:**

1. **맥락 기반 임베딩이 단일 기법으로는 가장 큰 개선**(+5~7%p)을 가져왔고, 청크에 문서 수준 맥락을 더하는 것이 검색 품질을 크게 높인다는 점을 확인해 줍니다. 이 기법 하나만으로도 최적 성능의 90% 지점까지 갈 수 있습니다.

2. **재순위화가 절대 성능은 가장 높아** Pass@10 95.26%에 도달합니다. 질의의 95%에서 올바른 청크가 상위 10개 안에 들어온다는 뜻입니다. 기준 RAG 대비 **검색 실패 47% 감소**(실패율 12.85% → 4.74%)에 해당합니다.

3. **트레이드오프가 중요합니다**: 각 기법은 복잡성과 비용을 더합니다.
   - 맥락 기반 임베딩: 일회성 적재 비용(프롬프트 캐싱 적용 시 이 데이터셋에 약 $3)
   - 하이브리드 검색: Elasticsearch 인프라와 유지보수 필요
   - 재순위화: 질의당 100~200ms 지연과 API 비용 추가(질의당 약 $0.002)

4. **요구 사항에 맞게 방식을 고르세요**:
   - **대량 처리, 비용 민감**: 맥락 기반 임베딩만(Pass@10 92%, 질의당 비용 없음)
   - **최대 정확도, 지연 허용**: 전체 재순위화 파이프라인(Pass@10 95%, 최고 정밀도)
   - **균형 잡힌 프로덕션 시스템**: 질의당 비용 없이 견고한 성능을 내는 하이브리드 검색(Pass@10 93%)

대부분의 프로덕션 RAG 시스템에는 **맥락 기반 임베딩이 성능 대비 비용이 가장 좋습니다.** 일회성 적재 비용만으로 Pass@10 92%를 냅니다. 하이브리드 검색과 재순위화는 그 2~3%p의 정밀도가 더 필요하고 추가 인프라나 질의 비용을 감당할 수 있을 때 선택하면 됩니다.

### 다음 단계와 핵심 정리

1) 맥락 기반 임베딩으로 검색 성능을 개선하는 방법을 보여 주고, 맥락 기반 BM25와 재순위화로 추가 개선을 이뤘습니다.

2) 이 예제는 코드베이스를 다뤘지만, 이 방법들은 사내 지식 베이스, 재무·법무 자료, 교육 콘텐츠 등 다른 데이터 유형에도 적용됩니다.

3) AWS 사용자라면 `contextual-rag-lambda-function`의 Lambda 함수로 시작할 수 있고, GCP 사용자라면 Cloud Run 인스턴스를 띄워 비슷한 패턴을 따르면 됩니다!